<!--
Copyright (c) 2026 OceanBase.

Licensed under the Apache License, Version 2.0 (the "License");
you may not use this file except in compliance with the License.
You may obtain a copy of the License at

http://www.apache.org/licenses/LICENSE-2.0

Unless required by applicable law or agreed to in writing, software
distributed under the License is distributed on an "AS IS" BASIS,
WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
See the License for the specific language governing permissions and
limitations under the License.
-->

# 19 · 让 Agent 主动调用记忆工具

Middleware 会替一次模型请求准备上下文；另一种接入方式是让 Agent 自己决定何时调用记忆工具。本篇先用独立进程建立真实 MCP 会话，再让真实模型选择 LangGraph 集成提供的工具。

需要支持工具调用的真实对话模型。我们只让模型保存本篇明确给出的合成项目规则，不让它猜测未经确认的长期约定。

路线：MCP 工具发现 → 通过 MCP 保存与搜索 → 真实 Agent 选择搜索与记忆工具 → 新会话验证 → 服务故障时的工具结果。

In [ ]:
import sys
from pathlib import Path

from _tutorial import Tutorial, show, table

from powercontext.http import CreateScopeRequest

if not Path("_tutorial.py").is_file():
    sys.path.insert(0, str(Path.cwd() / "examples" / "jupyter"))
if previous_lab := globals().get("lab"):
    await previous_lab.close()
lab = await Tutorial.start("19", features=())
client = lab.client
assert client is not None
scope = await client.create_scope(
    CreateScopeRequest(
        title="订单 CSV 导入器 · 19", summary="本次教学实验的独立材料", idempotency_key=f"{lab.run_id}:main"
    )
)
scope_id = scope.scope_id

## 独立进程通过 MCP 连接

启用 Server 的 MCP 路径。MCP 的工具集合有明确边界，不等于全部 HTTP API。下面的业务请求写在单元格里，辅助进程只负责连接与序列化。

In [ ]:
import asyncio
import json
from uuid import uuid4

from powercontext.server.settings import McpConfig

lab.settings_overrides["mcp"] = McpConfig(enabled=True)
await lab.restart()
client = lab.client
marker = "CSV-" + uuid4().hex[:10]
worker_path = Path(sys.modules["_tutorial"].__file__).parent / "support" / "mcp_worker.py"
operations = [
    {
        "name": "remember_memory",
        "arguments": {
            "scope_id": scope_id,
            "kind": "decision",
            "text": f"release_codename: 本项目发布代号是 {marker}。",
            "reason": "教学用户已确认的项目事实",
        },
    },
    {"name": "search_memory", "arguments": {"scope_id": scope_id, "query": "release_codename", "limit": 3}},
]
process = await asyncio.create_subprocess_exec(
    sys.executable,
    str(worker_path),
    stdin=asyncio.subprocess.PIPE,
    stdout=asyncio.subprocess.PIPE,
    stderr=asyncio.subprocess.PIPE,
)
stdout, stderr = await asyncio.wait_for(
    process.communicate(json.dumps({"url": lab.base_url + "/mcp/", "calls": operations}).encode()), 60
)
assert process.returncode == 0, "MCP 子进程执行失败"
mcp_result = json.loads(stdout)
assert {"remember_memory", "search_memory", "continue_handoff"} <= set(mcp_result["tools"])
assert marker in json.dumps(mcp_result["calls"], ensure_ascii=False)
show({"发现工具数": len(mcp_result["tools"]), "调用": mcp_result["calls"]})

## 让真实模型选择工具

现在切换到正式 LangGraph 集成。模型需要自己发出 tool_calls；我们检查调用轨迹，再独立读取服务端结果。代码没有替模型直接写入第二条规则。

In [ ]:
from _tutorial import chat_settings
from langchain.agents import create_agent
from langchain_openai import ChatOpenAI
from powercontext_langgraph import PowerContextScope, powercontext_tools

agent = create_agent(
    ChatOpenAI(**chat_settings()),
    tools=powercontext_tools(),
    context_schema=PowerContextScope,
    system_prompt="You are the CSV project assistant. Use tools for project facts. Save only explicitly confirmed durable rules. Never claim a tool operation succeeded unless its result confirms it.",
)
agent_context = PowerContextScope(scope_id=scope_id, base_url=lab.base_url, timeout=10)
result = await agent.ainvoke(
    {
        "messages": [
            (
                "user",
                "先搜索 release_codename 并完整告诉我发布代号；然后记住我确认的规则：batch_limit: 每批最多 137 条订单。请实际调用工具完成两步。",
            )
        ]
    },
    context=agent_context,
)
calls = [call for message in result["messages"] for call in getattr(message, "tool_calls", [])]
assert {"powercontext_search", "powercontext_remember"} <= {call["name"] for call in calls}
assert marker in result["messages"][-1].text
table([{"工具": call["name"], "输入": call["args"]} for call in calls])
print(result["messages"][-1].text)

## 一个新会话独立读回

我们丢弃上一轮消息，只问每批数量。模型需要再次搜索已有 Memory；这个结果也能通过公开 Client 读取来复查。

In [ ]:
from powercontext.http import SearchMemoryRequest

stored = await client.search_memory(SearchMemoryRequest(scope_id=scope_id, query="batch_limit"))
assert any("137" in hit.text for hit in stored.hits)
next_run = await agent.ainvoke(
    {"messages": [("user", "请用工具查询 batch_limit。本项目每批最多多少条订单？")]}, context=agent_context
)
assert any(getattr(message, "tool_calls", []) for message in next_run["messages"])
assert "137" in next_run["messages"][-1].text
print(next_run["messages"][-1].text)

## 练习与验收

把 context 的 scope_id 换成新项目，再问相同问题，预期没有 137 的依据。自动注入见第 10 篇，失败时工具如何明确返回不可用见第 21 篇。

接下来阅读 [20_team_dashboard_reports.ipynb](20_team_dashboard_reports.ipynb)。

最后关闭服务。实验文件保留在本次 `.powercontext/` 目录，便于复查。

In [ ]:
await lab.close()
print("本篇 Server 已关闭。")